### Tauha Imran 22i1239 - G4
#### NLP - A2 - Vectors & Language Modelling
---

### 1. OCR and Preprocessing

Extracting text from PDF judgements using:

● pdf2image → to convert PDF pages to images

● pytesseract or EasyOCR → to extract text from images


Clean and normalize the text: remove headers, footers, and line breaks; fix
hyphenation and punctuation; split into sentences.
Save each record in JSON format:
```
{
"case_id": "SCP_2025_001",
"pdf_source": "path/to/file.pdf",
"ocr_text": "Full extracted text..."
}
```

In [ ]:
# install pre req libraries
%pip install pdf2image pytesseract pillow nltk
%pip install pdf2image pytesseract easyocr nltk spacy
%pip install sklearn


In [2]:
#including libraries and downloading necessary data

import os, json, re
import numpy as np
from pdf2image import convert_from_path #using this to pfd -> image conversion
import pytesseract # THE OCR TOOL OF MY CHOICE
import nltk, spacy
from nltk.tokenize import sent_tokenize, word_tokenize #tokenizer functions
nltk.download('punkt')

#PS - i installed tesseract-ocr in my system using https://github.com/UB-Mannheim/tesseract/wiki
#and added the path to system environment variables C:\Program Files\Tesseract-OCR
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
#  the line above kinda did the environment variable setting lol..

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# Ensure required NLTK tokenizer resources are available in a local nltk_data directory
import nltk, os
nltk_data_dir = os.path.join(os.getcwd(), 'nltk_data')
os.makedirs(nltk_data_dir, exist_ok=True)
# Add our local nltk_data directory to nltk lookup paths if not already present
if nltk_data_dir not in nltk.data.path:
    nltk.data.path.append(nltk_data_dir)
# Some environments require both 'punkt' and 'punkt_tab' tokenizers (punkt_tab contains language-specific tables)
required = {'punkt':'tokenizers/punkt', 'punkt_tab':'tokenizers/punkt_tab'}
for pkg, path in required.items():
    try:
        nltk.data.find(path)
        print(f"NLTK resource '{pkg}' already present ({path})")
    except LookupError:
        print(f"Downloading NLTK resource: {pkg} to {nltk_data_dir}...")
        nltk.download(pkg, download_dir=nltk_data_dir)
print('NLTK data path:', nltk_data_dir)
print('nltk.data.path:', nltk.data.path)

NLTK resource 'punkt' already present (tokenizers/punkt)
NLTK resource 'punkt_tab' already present (tokenizers/punkt_tab)
NLTK data path: f:\Projects\NLPlayground\Assignment2_Vectors & Language Modeling\nltk_data
nltk.data.path: ['C:\\Users\\LENOVO/nltk_data', 'c:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python313\\nltk_data', 'c:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python313\\share\\nltk_data', 'c:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python313\\lib\\nltk_data', 'C:\\Users\\LENOVO\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'f:\\Projects\\NLPlayground\\Assignment2_Vectors & Language Modeling\\nltk_data']


In [4]:
from pdf2image import convert_from_path
import pytesseract
import json
import re

# function to convert pdf to text using OCR
def pdf_to_text(pdf_path):
    pages = convert_from_path(pdf_path, dpi=300) # <<---- this converts each page to an image
    text = ""
    for page in pages:
        text += pytesseract.image_to_string(page)
    return text

def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'-\s+', '', text)  # fix hyphenation
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

pdf_path = "data/sample_case.pdf"
raw_text = pdf_to_text(pdf_path)
cleaned_text = clean_text(raw_text)

record = {
    "case_id": "SCP_2025_001",
    "pdf_source": pdf_path,
    "ocr_text": cleaned_text
}

with open("outputs/ocr_data.json", "w", encoding="utf-8") as f:
    json.dump(record, f, indent=4)

#print some stats
print(f"Extracted {len(cleaned_text)} characters from the PDF.")

Extracted 7601 characters from the PDF.


In [5]:
#print some stats
print(f"Extracted {len(cleaned_text)} characters from the PDF.")

Extracted 7601 characters from the PDF.


---

### 2. Neural Language Model for Sentence Embeddings
Now we build a Neural Language Model (from scratch) to learn distributed vector representations (embeddings) of sentences.

1. Data Preparation
Use your provided legal text corpus.
Example:

The court found the defendant guilty.
The defendant appealed the decision.
The judge dismissed the case.

1. Tokenize text into words.
2. Build a vocabulary of unique words.
3. Create training samples depending on your chosen model:
o CBOW: Predict a word from the surrounding context.
o Skip-gram: Predict surrounding context words from a given target
word.

2. Model Implementation (Using NumPy Only)

A minimal architecture:
Input → Hidden Layer (Word Embeddings) → Output (Softmax)

Example initialisation:
vocab_size = len(vocab)
embedding_dim = 50
W1 = np.random.randn(vocab_size, embedding_dim)
W2 = np.random.randn(embedding_dim, vocab_size)
Train using cross-entropy loss and gradient descent to predict target
words.

In [6]:
import json
import re
import nltk # natural language toolkit
from nltk.tokenize import sent_tokenize, word_tokenize  #functions that i'mma use for tokenization
nltk.download('punkt')  # downloading the punkt tokenizer models

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [7]:
# STEP1 -> Loading the ocr data
# now getting the data from the .json file and loading it into memory for our nueral network model.

#LOAD THE OCR DATA FROM JSON
with open("outputs/ocr_data.json", "r", encoding="utf-8") as f:
    record = json.load(f)

text = record["ocr_text"].lower() #making it all lowercase

#splitting it into sentences.
sentences = sent_tokenize(text)
print(f"Total sentences extracted: {len(sentences)}")


Total sentences extracted: 33


In [8]:
# STEP2 -> Tokenie & Build Vocabulary

#tokenizing all the words
tokens = word_tokenize(text)
print(f"Total tokens extracted: {len(tokens)}")

#removing punctuation and non-alphabetic tokens
tokens = [token for token in tokens if token.isalpha()]
print(f"Total alphabetic tokens after cleaning: {len(tokens)}")


#building vocabulary
vocab = sorted(set(tokens))
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx:word for word,idx in word2idx.items()}

print(f"Vocabulary size: {len(vocab)}") #printing vocab size

Total tokens extracted: 1377
Total alphabetic tokens after cleaning: 1151
Vocabulary size: 360


In [ ]:
#  STEP 3 - Generate Training Data ( Skip-gram pairs)
import random

def generate_skipgram_pairs(tokens, window_size=2):
    pairs = []
    for i, target_word in enumerate(tokens):
        context_indices = list(range(max(0, i - window_size), min(len(tokens), i + window_size + 1)))
        context_indices.remove(i)  # remove the target word index
        for j in context_indices:
            context_word = tokens[j]
            pairs.append((target_word, context_word))
    return pairs

pairs = generate_skipgram_pairs(tokens, window_size=2)
print(f"Total skip-gram pairs generated: {len(pairs)}")

#to verify
#random.sample(pairs, 5)


Total skip-gram pairs generated: 4598


[('comments', 'submitted'),
 ('declared', 'as'),
 ('by', 'the'),
 ('of', 'different'),
 ('other', 'categories')]

In [14]:
# STEP4 -> Initializing the nueral network model using Numpy only
import numpy as np

v = len(vocab)  #vocab size
embedding_dim = 100  #dimension of the embedding vector

# randomly initializing weight matrices
W1 = np.random.rand(v, embedding_dim)  #input to hidden
W2 = np.random.rand(embedding_dim, v)  #hidden to output

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

In [15]:
# STEP 5 -> the training loop

def one_hot_encode(word, word2idx, vocab_size):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1
    return vec

def train_skipgram(pairs, epochs = 5 , learning_rate = 0.05):
    global W1, W2
    v = len(vocab) #vocab size
    for epoch in range(epochs):
        loss = 0
        for target_word, context_word in random.sample(pairs, 500): # using a subset of 500 for faster training
            target_idx = word2idx[target_word]
            context_idx = word2idx[context_word]
            
            x = one_hot_encode(target_word, word2idx, v)

            #forward pass
            h = np.dot(W1.T, x)  #hidden layer
            u = np.dot(W2.T, h)  #output layer 
            y_pred = softmax(u)  #predicted output

            #compute loss (cross-entropy)
            loss += -np.log(y_pred[context_idx] + 1e-9)

            #backpropagation / backward pass / weight updates
            e = y_pred
            e[context_idx] -= 1
            dW2 = np.outer(h, e)
            dW1 = np.outer(x, np.dot(W2, e))
            
            # update weights
            W1 -= learning_rate * dW1
            W2 -= learning_rate * dW2

        # print average loss per epoch
        print(f"Epoch {epoch+1}, Loss: {loss/500:.4f}")


############################################
# Training the model
train_skipgram(pairs, epochs=5, learning_rate=0.05)



Epoch 1, Loss: 6.0227
Epoch 2, Loss: 5.5488
Epoch 3, Loss: 5.4283
Epoch 4, Loss: 5.3768
Epoch 3, Loss: 5.4283
Epoch 4, Loss: 5.3768
Epoch 5, Loss: 5.1344
Epoch 5, Loss: 5.1344


In [16]:
# STEP 6 -> Generating Sentence Embeddings
def get_sentence_embedding(sentence):
    words = [w for w in word_tokenize(sentence.lower()) if w in word2idx]
    if not words:
        return np.zeros(embedding_dim)
    vectors = [W1[word2idx[w]] for w in words]
    return np.mean(vectors, axis=0)

sentence_vectors = [get_sentence_embedding(s) for s in sentences]
print(f"Computed {len(sentence_vectors)} sentence embeddings.")


Computed 33 sentence embeddings.


---

#### 3. Extractive Summarization

Once sentence embeddings are computed:
1. Compute a document-level embedding as the mean of all sentence vectors.
2. Measure sentence importance using cosine similarity between each sentence
vector and the document vector.
3. Rank sentences by importance score.
4. Select top-k sentences for the summary.
Each training sample should contain:

```
{
"input": "Full OCR text",
"target": "Summarised text"
}
```

Output Example.

```
Original:
The court found the defendant guilty. The defendant appealed the decision. The judge
dismissed the case due to insufficient evidence.

Extractive Summary (Top 2 Sentences):
The court found the defendant guilty. The judge dismissed the case due to insufficient
evidence.
```

In [ ]:
# Here's the code for the extractive summarization using cosine similarity
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity # just the cosine similarity function

def extractive_summary(sentences, sentence_vectors, top_n=5):
    sim_matrix = cosine_similarity(sentence_vectors)
    scores = sim_matrix.sum(axis=1)
    ranked_sentences = [sentences[i] for i in np.argsort(scores)[-top_n:][::-1]]
    return ranked_sentences